![image.png](https://i.imgur.com/a3uAqnb.png)

# **🌊 Flood Area Segmentation with U-Net (Pretrained Encoder)**  
In this exercise, we will:

✅ **Build a custom Dataset class** for flood segmentation  
✅ **Use `segmentation_models_pytorch (SMP)`** to load **U-Net** with a **pretrained encoder**  
✅ Train the model and evaluate its performance 

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("faizalkarim/flood-area-segmentation")

print("Path to dataset files:", path)

## **1️⃣ Dataset Class**

- The dataset consists of:
- **Images:** RGB flood images (`.jpg`)
- **Masks:** Corresponding segmentation masks (`.png`)
- **Metadata:** A CSV file (`metadata.csv`) mapping images to masks.
- The masks highlight the **water regions** in the images.
- We will create a **custom PyTorch Dataset class** to load images and masks.





In [ ]:
# This function will be used to remap the mask values to either 0 or 1, use it in the custom dataset class

def remap_mask_binary(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0 (Use if Binary Segmentation)
    mask_np = mask.numpy().squeeze() 
    # Convert to binary: non-zero values become 1
    binary_mask = (mask_np != 0).astype(np.uint8)
    return torch.from_numpy(binary_mask).unsqueeze(0)

# Write your dataset class here

In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import numpy as np
# Custom Dataset Class
class FloodSegmentationDataset(Dataset):
    def __init__(self, root_dir, csv_file, transform=None, target_transform=None):
        TODO

    def __len__(self):
        TODO

    def __getitem__(self, idx):
        TODO


In [ ]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

# Define transforms for images and masks
image_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((256, 256)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Standard ImageNet normalization
])

mask_transforms = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),  # Keep segmentation masks intact
    transforms.PILToTensor(),

])

In [ ]:
## **🔹 Splitting the Dataset into Train & Test**

# Define dataset paths
csv_path = os.path.join(path, "metadata.csv")

# Read metadata CSV
metadata = pd.read_csv(csv_path)

# Split dataset into 80% train, 20% test
train_data, test_data = train_test_split(metadata, test_size=0.2, random_state=42, shuffle=True)

# Save split CSVs
train_data.to_csv(os.path.join(path, "train.csv"), index=False)
test_data.to_csv(os.path.join(path, "test.csv"), index=False)

In [ ]:
# Load training dataset
train_dataset = FloodSegmentationDataset(root_dir=path, csv_file=os.path.join(path, "train.csv"),
                                         transform=image_transforms, target_transform=mask_transforms)

# Load testing dataset
test_dataset = FloodSegmentationDataset(root_dir=path, csv_file=os.path.join(path, "test.csv"),
                                        transform=image_transforms, target_transform=mask_transforms)


# Create Train & Test DataLoaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)

# Check dataset sizes
print(f"Training Samples: {len(train_dataset)}, Testing Samples: {len(test_dataset)}")


# Display some images from the dataset

In [ ]:
TODO

## **2️⃣ Model Class**

- We use **U-Net** from the **`segmentation_models_pytorch (SMP)`** library.
- The encoder (backbone) is **pretrained EfficientNet-B0** for better feature extraction.
- The decoder is **randomly initialized** and trained for segmentation.
- **Binary segmentation output (1 class for flood area)**  
- **Sigmoid activation** to output probability maps  
- **Binary Cross Entropy (BCE) Loss**  

![image.png](https://i.imgur.com/UVgm5kz.png)

In [ ]:
!pip install -q segmentation_models_pytorch

In [ ]:
import segmentation_models_pytorch as smp

# Define U-Net Model
device = "cpu"
model = smp.Unet(
    encoder_name=TODO,  # Pretrained encoder (backbone) Hint ^
    encoder_weights="imagenet",  # Use ImageNet weights
    in_channels=TODO,
    classes=TODO,
).to(device)

## **3️⃣ Training and Validation Loops**

In [ ]:
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader):
        images, masks = images.to(device), masks.to(device).to(torch.float)
        
        outputs = model(images)  
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device), masks.to(device).to(torch.float)

            outputs = model(images)  
            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / len(dataloader)

## **4️⃣ Running Training**
- We fine-tune the **pretrained U-Net model** on our flood segmentation dataset.
- We train the model using **Binary Cross Entropy (BCE) Loss**.

In [ ]:
import torch
from torch import nn
# Define loss function and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.0001)

num_epochs = 10  # Define number of epochs
train_losses = []
val_losses = []

# Training Loop
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")


### **🔹 Plot Training Loss Curve**


In [ ]:
import matplotlib.pyplot as plt

plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()

## **5️⃣ Visualizing Predictions**

- We compare **predicted flood masks** against **ground truth masks**.
- We visualize results for multiple test images.

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np

# Function to denormalize images
def denormalize(img):
    mean = np.array([0.485, 0.456, 0.406])  # ImageNet mean
    std = np.array([0.229, 0.224, 0.225])  # ImageNet std
    img = img.numpy().transpose(1, 2, 0)  # Convert to HWC
    img = img * std + mean  # Reverse normalization
    img = np.clip(img, 0, 1)  # Clip values to [0,1]
    return img

# Set model to evaluation mode
model.eval()

# Get some test samples
test_samples = random.sample(range(len(test_dataset)), 5)

for idx in test_samples:
    img, mask = test_dataset[idx]

    with torch.no_grad():
        pred_mask = model(img.unsqueeze(0).to(device))  # Forward pass

    pred_mask = (pred_mask >= 0.5).cpu().squeeze().numpy()  # Convert to binary mask

    # Display images
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Original Image (Denormalized)
    axes[0].imshow(denormalize(img))
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    # Ground Truth Mask
    axes[1].imshow(mask.squeeze(), cmap="gray")
    axes[1].set_title("Ground Truth Mask")
    axes[1].axis("off")

    # Predicted Mask
    axes[2].imshow(pred_mask, cmap="gray")
    axes[2].set_title("Predicted Mask")
    axes[2].axis("off")

    plt.show()


### Contributed by: Mohamed Eltayeb